# Tahap 3: Uji Coba Model dengan Data Baru (Inference)
Notebook ini berfungsi sebagai lingkungan simulasi untuk menguji model Random Forest yang telah disimpan di dalam *registry* MLflow.

Fokus utama pengujian ini adalah melakukan proses *Inference* (Prediksi) terhadap data pasien baru yang belum pernah dilihat oleh model. Selain itu, notebook ini mensimulasikan alur kerja *backend* Aplikasi Web, di mana data mentah (Berat dan Tinggi Awal/Akhir) akan dihitung nilai `Kecepatan_Tumbuh`-nya secara dinamis sebelum dimasukkan ke dalam model prediksi.

In [1]:
import pandas as pd
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

# 1. Load Model secara otomatis dari eksperimen terakhir MLflow
mlflow.set_tracking_uri("file:../mlruns")
experiment = mlflow.get_experiment_by_name('Stunting_Dynamic_Model_Experiment')
latest_run = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=['start_time DESC']).iloc[0]
run_id = latest_run.run_id

print(f"Memuat model cerdas dari Run ID: {run_id}")
model = mlflow.sklearn.load_model(f"runs:/{run_id}/dynamic_rf_model")

c:\Users\rianp\miniconda3\envs\stunting_ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Memuat model cerdas dari Run ID: 51226154168f45ae86a452f779336052


### 1. Buat Data Pasien Baru

In [2]:
# Pasien A: Pertumbuhan sangat sehat dan cepat (Aman)
# Pasien B: Pertumbuhan terhambat, tinggi badan nyaris tidak naik (Risiko Stunting)

data_baru = pd.DataFrame({
    'Umur (Bulan)': [32, 10],            # Sama-sama umur 1 tahun
    'Jenis Kelamin': [1, 0],             # 1: L, 0: P
    'BB_Awal': [9.0, 8.0],
    'TB_Awal': [83.2, 72.0],             # Titik awal sama persis!
    'BB_Akhir': [10.1, 8.0],              # Pasien A naik 1.5kg, Pasien B cuma naik 0.1kg
    'TB_Akhir': [83.2, 72.0],            # Pasien A nambah 4cm, Pasien B cuma 0.5cm (Gagal Tumbuh)
    'Lama_Pantau_Bulan': [5, 3]          # Sama-sama dipantau selama 4 bulan
})

# Preprocessing (Sistem Backend Web akan otomatis menghitung ini nanti)
interval = data_baru['Lama_Pantau_Bulan'] - 1
data_baru['Kecepatan_Tumbuh_BB'] = (data_baru['BB_Akhir'] - data_baru['BB_Awal']) / interval
data_baru['Kecepatan_Tumbuh_TB'] = (data_baru['TB_Akhir'] - data_baru['TB_Awal']) / interval
data_baru['Rasio_BB_TB_Akhir'] = data_baru['BB_Akhir'] / data_baru['TB_Akhir']

print("Tabel Data yang masuk ke Model:")
data_baru

Tabel Data yang masuk ke Model:


,Umur (Bulan),Jenis Kelamin,BB_Awal,TB_Awal,BB_Akhir,TB_Akhir,Lama_Pantau_Bulan,Kecepatan_Tumbuh_BB,Kecepatan_Tumbuh_TB,Rasio_BB_TB_Akhir
0,32,1,9.0,83.2,10.1,83.2,5,0.275,0.0,0.121394
1,10,0,8.0,72.0,8.0,72.0,3,0.000,0.0,0.111111


### 2. Lakukan Prediksi

In [3]:
prediksi = model.predict(data_baru)

# Tampilkan Hasil
data_baru['Status_Prediksi'] = prediksi
data_baru['Kesimpulan (Model)'] = data_baru['Status_Prediksi'].map({0: 'NORMAL', 1: 'STUNTING'})

print("\n--- HASIL DETEKSI MODEL ---")
data_baru[['TB_Awal', 'TB_Akhir', 'Kecepatan_Tumbuh_TB', 'Kesimpulan (Model)']]


--- HASIL DETEKSI MODEL ---


,TB_Awal,TB_Akhir,Kecepatan_Tumbuh_TB,Kesimpulan (Model)
0,83.2,83.2,0.0,STUNTING
1,72.0,72.0,0.0,NORMAL
